# NB05 - Curriculum Learning
## Background
Curriculum learning (Bengio et al., 2009) was inspired by how humans learn: easier concepts before harder ones. Training on easy examples first gives the model a stable initial solution before introducing complexity. This improves both convergence speed and final accuracy, particularly when the data has high label noise — easy examples provide reliable gradients.

Self-paced learning (Kumar et al., 2010) automates the easy-to-hard ordering: the model dynamically selects which examples are currently 'learnable' based on per-sample loss. Baby steps curriculum (Cirik et al., 2016) interpolates between random and fully sorted training throughout training. Modern LLM pretraining uses implicit curriculum: web data is filtered for quality first, then mixed with higher-quality sources.

## What You'll Learn
- Difficulty scoring: model loss as a proxy for example difficulty
- Curriculum ordering: sorted, paced, and random baselines
- Self-paced learning: automatic difficulty selection threshold
- Training curves under different curriculum regimes

## Why This Matters
Curriculum learning is most impactful when data quality is heterogeneous. When training on web-scraped text or noisy annotations, curriculum ordering can significantly accelerate convergence and reduce the impact of mislabeled hard negatives.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple

np.random.seed(42)

# ── Dataset: easy and hard examples ───────────────────────────────────────
n_easy, n_hard, n_noisy = 400, 300, 100
n_features, n_classes = 6, 4

class_means = np.eye(n_classes, n_features) * 4  # Well-separated classes

def gen_class_samples(n, class_id, spread=1.0):
    return class_means[class_id] + np.random.randn(n, n_features) * spread

# Easy: tight clusters
X_easy = np.vstack([gen_class_samples(n_easy//n_classes, c, 0.5) for c in range(n_classes)])
y_easy = np.repeat(np.arange(n_classes), n_easy//n_classes)

# Hard: overlapping clusters
X_hard = np.vstack([gen_class_samples(n_hard//n_classes, c, 2.0) for c in range(n_classes)])
y_hard = np.repeat(np.arange(n_classes), n_hard//n_classes)

# Noisy: mislabeled
X_noisy = np.vstack([gen_class_samples(n_noisy//n_classes, c, 1.0) for c in range(n_classes)])
y_noisy = (np.repeat(np.arange(n_classes), n_noisy//n_classes) + 1) % n_classes  # Wrong labels

X_all = np.vstack([X_easy, X_hard, X_noisy])
y_all = np.concatenate([y_easy, y_hard, y_noisy])
difficulty = np.concatenate([
    np.zeros(len(X_easy)),      # easy
    np.ones(len(X_hard)),       # hard
    np.ones(len(X_noisy)) * 2,  # noisy/corrupted
])

n_total = len(X_all)
print(f'Dataset: {n_total} samples ({n_easy} easy, {n_hard} hard, {n_noisy} noisy)')

# Test set
test_idx = np.random.choice(n_total, 150, replace=False)
train_idx = np.setdiff1d(np.arange(n_total), test_idx)
X_test, y_test = X_all[test_idx], y_all[test_idx]
# Clean test set (no noisy labels)
clean_test_mask = difficulty[test_idx] < 2
X_test_clean = X_test[clean_test_mask]
y_test_clean = y_test[clean_test_mask]

In [ ]:
# ── Classifier for curriculum experiments ─────────────────────────────────
class SoftmaxClf:
    def __init__(self, n_features, n_classes, lr=0.1, seed=0):
        np.random.seed(seed)
        self.W = np.random.randn(n_features, n_classes) * 0.1
        self.b = np.zeros(n_classes)
        self.lr = lr

    def softmax(self, X):
        z = X @ self.W + self.b
        z -= z.max(axis=1, keepdims=True)
        e = np.exp(z)
        return e / e.sum(axis=1, keepdims=True)

    def per_sample_loss(self, X, y):
        probs = self.softmax(X)
        return -np.log(probs[np.arange(len(y)), y] + 1e-10)

    def update(self, X, y):
        probs = self.softmax(X)
        n = len(y)
        dz = probs.copy(); dz[np.arange(n), y] -= 1; dz /= n
        self.W -= self.lr * X.T @ dz
        self.b -= self.lr * dz.sum(axis=0)
        return -np.mean(np.log(probs[np.arange(n), y] + 1e-10))

    def accuracy(self, X, y): return np.mean(np.argmax(self.softmax(X), axis=1) == y)

# ── Curriculum strategies ──────────────────────────────────────────────────
def compute_difficulty_scores(model, X, y):
    return model.per_sample_loss(X, y)

def run_curriculum(strategy: str, n_epochs: int = 50, batch_size: int = 32,
                   pacing_fn=None) -> List[float]:
    model = SoftmaxClf(n_features, n_classes, lr=0.05)
    X_tr, y_tr = X_all[train_idx], y_all[train_idx]
    n_tr = len(X_tr)
    test_accs = []

    for epoch in range(n_epochs):
        progress = epoch / n_epochs

        if strategy == 'random':
            order = np.random.permutation(n_tr)
        elif strategy == 'easy_first':
            # Sort by pre-computed difficulty (loss on untrained model)
            losses = compute_difficulty_scores(model, X_tr, y_tr)
            order = np.argsort(losses)  # Easiest first
        elif strategy == 'hard_first':
            losses = compute_difficulty_scores(model, X_tr, y_tr)
            order = np.argsort(losses)[::-1]  # Hardest first
        elif strategy == 'self_paced':
            # Include only examples with loss below threshold (auto-paced)
            losses = compute_difficulty_scores(model, X_tr, y_tr)
            threshold = np.quantile(losses, 0.3 + 0.7 * progress)  # Grow threshold
            easy_idx = np.where(losses <= threshold)[0]
            order = easy_idx[np.random.permutation(len(easy_idx))]

        for i in range(0, len(order), batch_size):
            batch = order[i:i+batch_size]
            model.update(X_tr[batch], y_tr[batch])

        test_accs.append(model.accuracy(X_test_clean, y_test_clean))

    return test_accs

print('Running curriculum experiments...')
strategies = ['random', 'easy_first', 'hard_first', 'self_paced']
all_accs = {}
for strat in strategies:
    accs = run_curriculum(strat)
    all_accs[strat] = accs
    print(f'  {strat:12s}: final acc = {accs[-1]:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
colors = {'random': 'gray', 'easy_first': 'blue', 'hard_first': 'red', 'self_paced': 'green'}
for strat, accs in all_accs.items():
    ax.plot(accs, label=strat, color=colors[strat], linewidth=1.5)
ax.set_title('Curriculum Learning: Test Accuracy (clean samples)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Test accuracy')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{BASE}/s29_05_curriculum.png', dpi=100, bbox_inches='tight')
plt.show()
print('Best strategy in the presence of noisy data: easy_first or self_paced')